# Part 2 : Cortex AI ハンズオン

## このノートブックでやること
1. **名寄せ（エンティティ解決）** — 完全一致 vs AI_SIMILARITY
2. **SNS ログの AI 分析** — AI_EXTRACT / AI_SENTIMENT / AI_CLASSIFY
3. **音声ログの AI 加工** — AI_REDACT / AI_CLASSIFY / AI_AGG

## 使用データ
GlacierStyle — 架空 EC サイトのサンプルデータセット


In [ ]:
-- コンテキスト設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;


---
## 1. 名寄せ（エンティティ解決）

仕入先から届いた商品リストと自社商品マスタを突合する処理です。
実務では「同じ商品なのに名前が微妙に違う」というケースが頻発します。

**表記揺れの例:**
- 全角/半角: `ＬＥＤデスクライト` → `LEDデスクライト`
- 同義語: `ライト` ↔ `ランプ` ↔ `灯`
- ブランド名付き: `GlacierStyle スタイリッシュデスクライト`
- カタカナ揺れ: `Tシャツ` → `ティーシャツ`
- 語順逆転: `ウール 高品質ブランケット` → `高品質ウールブランケット`

| 難易度 | 割合 | パターン |
|---|---|---|
| Easy | 〜20% | 全角英数字、スペース挿入 |
| Medium | 〜30% | 同義語、ブランド名付加 |
| Hard | 〜30% | 型番付き、英語名のみ、語順逆転 |
| Very Hard | 〜15% | 完全に異なる表現、OCR風誤字 |
| False Positive | 〜5% | マッチすべきでない商品 |


In [ ]:
-- 仕入先商品テーブルの作成（表記揺れを含む100件のデータ）
CREATE OR REPLACE TABLE supplier_products_v2 (
    supplier_product_id   VARCHAR PRIMARY KEY,
    supplier_product_name VARCHAR,
    supplier_name         VARCHAR,
    supplier_price        DECIMAL(10,2),
    supplier_category     VARCHAR,
    original_product_id   VARCHAR  -- 正解データ（精度検証用）
);

COPY INTO supplier_products_v2
FROM @DATA_STAGE/supplier_products_v2.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
ON_ERROR = 'CONTINUE';


In [ ]:
-- 仕入先データのプレビュー
SELECT
    supplier_product_id,
    supplier_product_name,
    supplier_name,
    supplier_price,
    supplier_category
FROM supplier_products_v2
LIMIT 10;


In [ ]:
-- 商品マスタのプレビュー（突合先）
SELECT
    product_id,
    product_name,
    category_l2,
    brand,
    current_price
FROM dim_products
LIMIT 10;


### Step 1: 完全一致 JOIN — まずは愚直にやってみる

表記揺れがあるとどの程度マッチしないかを確認します。


In [ ]:
-- 完全一致 JOIN でどの程度マッチするか確認
WITH exact_match AS (
    SELECT
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name
    FROM supplier_products_v2 sp
    LEFT JOIN dim_products dp
        ON sp.supplier_product_name = dp.product_name
)
SELECT
    COUNT(*)                                            AS total_supplier_products,
    COUNT(product_id)                                   AS exact_matched,
    COUNT(*) - COUNT(product_id)                        AS unmatched,
    ROUND(COUNT(product_id) / COUNT(*) * 100, 1)        AS match_rate_pct
FROM exact_match;


In [ ]:
-- マッチしなかったレコードの例を確認
-- → 「これは同じ商品なのに一致しない」というケースが見える
SELECT
    sp.supplier_product_name AS "仕入先商品名",
    dp.product_name          AS "マスタ商品名（完全一致なし）"
FROM supplier_products_v2 sp
LEFT JOIN dim_products dp
    ON sp.supplier_product_name = dp.product_name
WHERE dp.product_id IS NULL
LIMIT 15;


### Step 2: AI_SIMILARITY — 意味で類似度を計算する

![AI_SIMILARITY](images/part2/ai_similarity.png)

`JAROWINKLER_SIMILARITY` などの従来の文字列類似度は**文字の並びしか見ない**ため、同義語（ライト↔ランプ）や語順逆転には対応できません。

`AI_SIMILARITY` は **AI が意味的な類似度を計算**するため、表記が全然違っても同じ意味なら高スコアになります。

| 比較 | JAROWINKLER | AI_SIMILARITY |
|---|---|---|
| `モダンデスクライト` vs `モダンデスクランプ` | 低（ランプ≠ライト） | 高（意味が近い） |
| `オーガニックコットンTシャツ` vs `有機綿ティーシャツ` | 低（文字列が全然違う） | 高（同じ商品） |


In [ ]:
-- AI_SIMILARITY vs JAROWINKLER の比較デモ
-- → AI_SIMILARITY は「意味」で類似度を計算するため、従来手法では拾えないパターンに強い
SELECT
    '文字列1'                                       AS name1,
    '文字列2'                                       AS name2,
    AI_SIMILARITY(name1, name2)                    AS ai_similarity,
    JAROWINKLER_SIMILARITY(name1, name2) / 100.0   AS jw_similarity
UNION ALL
SELECT 'モダンデスクライト',     'モダンデスクランプ',
    AI_SIMILARITY('モダンデスクライト', 'モダンデスクランプ'),
    JAROWINKLER_SIMILARITY('モダンデスクライト', 'モダンデスクランプ') / 100.0
UNION ALL
SELECT 'プレミアムカシミヤセーター', '高級カシミアニット',
    AI_SIMILARITY('プレミアムカシミヤセーター', '高級カシミアニット'),
    JAROWINKLER_SIMILARITY('プレミアムカシミヤセーター', '高級カシミアニット') / 100.0
UNION ALL
SELECT 'ワイヤレスヘッドホン',   '無線ヘッドフォン',
    AI_SIMILARITY('ワイヤレスヘッドホン', '無線ヘッドフォン'),
    JAROWINKLER_SIMILARITY('ワイヤレスヘッドホン', '無線ヘッドフォン') / 100.0
UNION ALL
SELECT 'オーガニックコットンTシャツ', '有機綿ティーシャツ',
    AI_SIMILARITY('オーガニックコットンTシャツ', '有機綿ティーシャツ'),
    JAROWINKLER_SIMILARITY('オーガニックコットンTシャツ', '有機綿ティーシャツ') / 100.0;


In [ ]:
-- AI_SIMILARITY による名寄せ（サンプル50件）
-- ※ 全件 CROSS JOIN は計算量が大きいためサンプルで実施
CREATE OR REPLACE TABLE work_ai_similarity_match AS
WITH sample_suppliers AS (
    SELECT * FROM supplier_products_v2 SAMPLE (50 ROWS)
),
similarity_calc AS (
    SELECT
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        AI_SIMILARITY(sp.supplier_product_name, dp.product_name) AS ai_sim,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id
            ORDER BY AI_SIMILARITY(sp.supplier_product_name, dp.product_name) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products dp
)
SELECT
    supplier_product_id,
    supplier_product_name,
    product_id   AS matched_product_id,
    product_name AS matched_product_name,
    ai_sim       AS ai_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認（スコアが低い順 = マッチが怪しいものを先頭に）
SELECT * FROM work_ai_similarity_match
ORDER BY ai_similarity ASC
LIMIT 20;


In [ ]:
-- 精度検証（正解データと突合）
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN ais.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_matches,
    ROUND(SUM(CASE WHEN ais.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END)
          / COUNT(*) * 100, 1) AS accuracy_pct
FROM work_ai_similarity_match ais
JOIN supplier_products_v2 sp ON ais.supplier_product_id = sp.supplier_product_id;


### 名寄せまとめ

| 手法 | 特徴 | 向いているケース |
|---|---|---|
| 完全一致 JOIN | コストゼロ・確実 | 表記が統一されているデータ |
| AI_SIMILARITY | 意味的な類似度・同義語に強い | 表記揺れが多いデータ |

**ポイント:** AI の判断は「傾向を掴む」用途に最適。個別の高精度判断が必要な場面では人手レビューと組み合わせる。


---
## 2. SNS ログの AI 分析

SNS メンション（Twitter / Instagram 等）の生データに AI 関数を適用します。

| 関数 | 処理内容 |
|---|---|
| `AI_EXTRACT` | テキストから商品名・カテゴリ・問い合わせタイプを抽出 |
| `AI_SENTIMENT` | 投稿の感情分析（ポジティブ / ネガティブ / ニュートラル） |
| `AI_CLASSIFY` | 投稿カテゴリ分類（称賛 / クレーム / 質問 / 提案） |

![AI_EXTRACT](images/part2/ai_extract.png)

![AI_SENTIMENT](images/part2/ai_sentiment.png)

![AI_CLASSIFY](images/part2/ai_classify.png)

**ポイント:** SELECT 文に関数を追加するだけで AI が動く。既存の SQL 資産をそのまま活用できる。


In [ ]:
-- SNS 生ログの AI 分析 → Gold 層テーブルへ保存
CREATE OR REPLACE TABLE gold_sns_mentions_analyzed AS
WITH extracted_data AS (
    SELECT
        post_id, platform, post_type, username, display_name,
        content, posted_at, likes, retweets, replies,
        hashtags, mentioned_products, media_urls,
        -- 商品名・カテゴリ・問い合わせタイプを抽出
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            content,
            OBJECT_CONSTRUCT(
                'product_name',  'mentioned product name',
                'category',      'product category (e.g., ファッション, インテリア, テック)',
                'inquiry_type',  'inquiry type (e.g., 質問, レビュー, クレーム, 称賛, 提案)'
            )
        ) AS extracted_info
    FROM raw_sns_mentions
),
sentiment_analysis AS (
    SELECT *,
        -- 感情分析
        SNOWFLAKE.CORTEX.AI_SENTIMENT(content) AS sentiment_result
    FROM extracted_data
),
classified_data AS (
    SELECT *,
        -- カテゴリ分類
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            content,
            ['称賛', 'クレーム', '質問', '提案']
        ) AS classification_result
    FROM sentiment_analysis
)
SELECT
    post_id, platform, post_type, username, display_name,
    content, posted_at, likes, retweets, replies,
    hashtags, mentioned_products, media_urls,
    -- 抽出結果
    extracted_info:response.product_name::VARCHAR  AS extracted_product_name,
    extracted_info:response.category::VARCHAR      AS extracted_category,
    extracted_info:response.inquiry_type::VARCHAR  AS inquiry_type,
    -- 感情分析結果
    sentiment_result:categories[0].name::VARCHAR   AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    -- 分類結果
    classification_result:labels[0]::VARCHAR       AS post_category,
    CURRENT_TIMESTAMP()                            AS processed_at
FROM classified_data;


In [ ]:
-- 結果確認
SELECT * FROM gold_sns_mentions_analyzed LIMIT 10;


---
## 3. 音声ログの AI 加工

コールセンター音声ログの文字起こしデータに AI 関数を適用します。

| 関数 | 処理内容 |
|---|---|
| `AI_REDACT` | 個人情報（氏名・電話番号・住所・クレカ番号）を自動マスキング |
| `AI_SENTIMENT` | 顧客感情の分析 |
| `AI_CLASSIFY` | 問い合わせカテゴリ分類 |
| `AI_AGG` | 通話内容を 400 文字以内に要約（GROUP BY 対応） |

![AI_REDACT](images/part2/ai_redact.png)

![AI_AGG](images/part2/ai_agg.png)

**金融向けポイント:** `AI_REDACT` は個人情報保護の観点から特に有効。通話ログをそのまま分析テーブルに入れるリスクを下げられる。


In [ ]:
-- 音声ログの AI 加工 → Gold 層テーブルへ保存
CREATE OR REPLACE TABLE gold_voice_logs AS
SELECT
    * EXCLUDE transcribed_text,

    -- 1. AI_REDACT: 個人情報を自動マスキング
    SNOWFLAKE.CORTEX.AI_REDACT(transcribed_text)::VARCHAR AS transcribed_text_masked,

    -- 2. AI_SENTIMENT: 顧客感情の分析（マスキング済みテキストを使用）
    SNOWFLAKE.CORTEX.AI_SENTIMENT(transcribed_text_masked) AS sentiment_result,
    sentiment_result:categories[0].name::VARCHAR            AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR       AS sentiment,

    -- 3. AI_CLASSIFY: 問い合わせカテゴリ分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        transcribed_text_masked,
        ['商品に関する問い合わせ', '配送に関する問い合わせ', '返品・交換',
         '決済・支払い', 'アカウント・会員登録', 'クレーム', 'その他']
    ) AS classification_result,
    classification_result:labels[0]::VARCHAR               AS inquiry_category,

    -- 4. AI_AGG: 通話内容を 400 文字以内に要約（GROUP BY 対応）
    AI_AGG(
        transcribed_text_masked,
        '音声ログを400文字以内で要約してください。顧客の主な問い合わせ内容、要望、および解決状況を含めてください。'
    ) AS transcribed_text_summary,

    CURRENT_TIMESTAMP() AS processed_at

FROM raw_voice_logs
GROUP BY ALL;


In [ ]:
-- 結果確認
SELECT * FROM gold_voice_logs LIMIT 10;


---
## まとめ

| やったこと | 使った関数 | ポイント |
|---|---|---|
| 名寄せ（完全一致） | JOIN | まずこれで問題を確認 |
| 名寄せ（AI） | `AI_SIMILARITY` | 同義語・語順逆転に強い |
| SNS 感情分析 | `AI_SENTIMENT` / `AI_CLASSIFY` | SELECT に追加するだけ |
| SNS 情報抽出 | `AI_EXTRACT` | 構造化されていないテキストから項目を取り出す |
| PII マスキング | `AI_REDACT` | 個人情報を自動検出して伏せ字に |
| 通話要約 | `AI_AGG` | GROUP BY 対応の集約 AI 関数 |

**次のパート (Part 3):** これらのデータを使って Snowflake Intelligence で自然言語分析を体験します。
